# Baseline Instance Segmentation: YOLO11s-seg

Notebook ini mengunduh Panoptic Indoor Segmentation v5 dari Roboflow, melatih YOLO11s-seg dengan pretrained weights, lalu mengevaluasi pada test set. Pengaturan sama dengan baseline YOLO11n: 50 epoch, image size 640, batch 16.

Di Kaggle, aktifkan GPU dan tambahkan Kaggle Secret bernama `ROBOFLOW_API_KEY` dengan API key Roboflow-mu.

In [1]:
!pip install -q ultralytics roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from pathlib import Path
from roboflow import Roboflow
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient
import yaml, torch, shutil

assert torch.cuda.is_available(), 'Aktifkan GPU di pengaturan Kaggle.'
print('GPU:', torch.cuda.get_device_name(0))
api_key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
project = rf.workspace('panopticindoor').project('panoptic-indoor-segmentation')
dataset = project.version(5).download('yolov11')
data_yaml = Path(dataset.location) / 'data.yaml'
with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)
print('Dataset:', dataset.location)
print('Splits:', {k: data_cfg.get(k) for k in ['train', 'val', 'test']})

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to panoptic-indoor-segmentation-5 in yolov11:: 100%|██████████| 1856/1856 [00:00<00:00, 7920.00it/s]

Dataset: /kaggle/working/panoptic-indoor-segmentation-5
Splits: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images'}


## Training YOLO11s-seg
Model dimulai dari pretrained weights `yolo11s-seg.pt`. Parameter training dibuat sama dengan YOLO11n agar hasilnya dapat dibandingkan.

In [3]:
model = YOLO('yolo11s-seg.pt')
train_result = model.train(
    data=str(data_yaml),
    task='segment',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='yolo11s_seg_baseline',
    exist_ok=True,
    plots=True,
    seed=42
)
run_dir = Path(model.trainer.save_dir)
best_weights = run_dir / 'weights' / 'best.pt'
print('Training folder:', run_dir)
print('Best weights:', best_weights)
print('Training log:', run_dir / 'results.csv')

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic-indoor-segmentation-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_seg_baseline, nbs=64

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/50      6.88G     0.9744      2.402      3.471      1.275          0        353        640: 100% ━━━━━━━━━━━━ 25/25 3.3it/s 7.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.2it/s 2.7s0.6ss
                   all        166       2021      0.437      0.239      0.206      0.161      0.423      0.241      0.202      0.134

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/50      6.89G     0.9189      2.091      2.633      1.233          0        336        640: 100% ━━━━━━━━━━━━ 25/25 3.4it/s 7.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.9it/s 3.1s0.7ss
                   all        166       2021      0.485       0.19      0.171      0.128      0.261      0.245       0.17      0.112



## Evaluasi pada test set
Evaluasi berikut memakai test set dataset Roboflow yang berisi 359 gambar.

In [4]:
assert data_cfg.get('test'), 'Split test tidak ditemukan di data.yaml.'
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(data=str(data_yaml), split='test', imgsz=640, device=0, plots=True)
test_report = (
    f"Evaluated split: test\n"
    f"Box mAP50-95: {test_metrics.box.map:.6f}\n"
    f"Mask mAP50: {test_metrics.seg.map50:.6f}\n"
    f"Mask mAP50-95: {test_metrics.seg.map:.6f}\n"
)
print(test_report)
(run_dir / 'test_metrics.txt').write_text(test_report)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s-seg summary (fused): 113 layers, 10,095,454 parameters, 0 gradients, 33.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1400.8±948.1 MB/s, size: 69.3 KB)
val: Scanning /kaggle/working/panoptic-indoor-segmentation-5/test/labels... 359 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 359/359 747.0it/s 0.5s0.1s
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/002_jpg.rf.941b44d4633817fd833de04127e2f3a3.jpg: 1 duplicate labels removed
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/236_jpeg.rf.ce43674993187cb554fcbd2c85146baf.jpg: 2 duplicate labels removed
val: New cache created: /kaggle/working/panoptic-indoor-segmentation-5/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s0.3s
                   all        359      

90

## Siapkan file untuk diunduh
Setelah training dan evaluasi selesai, unduh `best.pt`, `results.csv`, dan `test_metrics.txt` dari folder output Kaggle.

In [5]:
export_dir = Path('/kaggle/working/yolo11s_baseline')
export_dir.mkdir(parents=True, exist_ok=True)
for filename in ['weights/best.pt', 'results.csv', 'test_metrics.txt']:
    source = run_dir / filename
    if source.exists():
        shutil.copy2(source, export_dir / Path(filename).name)
    else:
        print('Tidak ditemukan:', source)
print('File siap diunduh:', sorted(p.name for p in export_dir.iterdir()))

File siap diunduh: ['best.pt', 'results.csv', 'test_metrics.txt']
